# M3 – Step 7: Oracle request 3 (last 6 points)

Same method and score definitions as request 2 (`src/m3_surrogate/select_points.py`), now fitted on the 23 unique
points of requests 1 + 2.

Split (my choice): **2 exploitation + 3 exploration + 1 random, no repeat** (noise ≈ 0.003 from the request-2 repeat,
so another repeat would add little).

- Exploitation targets: the **snapped** M1 designs A–E after the re-run with the cubic-RBF FM
  (`results/m1_fm_hover_points.csv`, loaded and empty hover thrust). Weight 1 for baseline C, 0.6 for the others.
- Exploration = 0.7·σ + 0.3·|RBF cubic − GP|, normalised. Min distance 0.10 (scaled) to every point. Random seed 6003.
- **Hold-out protocol:** before sending, every model's prediction at the 6 points is frozen to
  `results/m3_request3_frozen_predictions.csv`; when `result_3.csv` arrives, score it against that file before refitting.

Write your **prediction before running** and **observation after** for each section.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from m3_surrogate import (load_oracle, average_repeats, make_features, write_request_csv, freeze_predictions,
                          QuadraticSurrogate, RBFSurrogate, GPSurrogate)
from m3_surrogate import select_points as sp

STUDENT_ID = 'ROLL_NUMBER'      # <-- your roll number before sending
ORACLE_DIR = ROOT / 'data' / 'oracle'; RES_DIR = ROOT / 'results'; FIG_DIR = ROOT / 'figures'
pd.set_option('display.width', 200)

raw = load_oracle(ORACLE_DIR)
assert set(raw.batch) == {'result_1', 'result_2'}, "planning request 3: only result_1 and result_2 should be loaded"
df, repeats = average_repeats(raw)
hover_all = pd.read_csv(RES_DIR / 'm1_fm_hover_points.csv')
hover = hover_all[hover_all.version == 'snapped']
print(f"{len(raw)} runs, {len(df)} unique points; {len(hover)} M1 hover points (snapped designs)")

## 1. How well do we already know FM where M1 needs it?

`sigma` = GP σ at each hover point, `dist_existing` = scaled distance to the nearest measured point.

Prediction: _..._

In [ ]:
X = make_features(df.D_m, df.T_N); y = df.FM.values
gp = GPSurrogate().fit(X, y)
h = sp.m1_hover_points(hover)
Xh = make_features(h.D_m, h.T_N)
h['sigma'] = gp.predict(Xh, return_std=True)[1]
h['dist_existing'] = np.linalg.norm(Xh[:, None] - X[None], axis=2).min(axis=1)
h.round(4)

Observation: _..._

## 2. Score and choose the 6 points

Prediction (where will exploration go now that the corners are measured? how large will the exploitation scores be
compared with request 2, where the top one was 0.47?): _..._

In [ ]:
plan, grid = sp.choose_request(df, hover, n_exploit=2, n_explore=3, n_random=1, repeat_pts=(),
                               seed=6003, label='R3')
print(f"largest exploitation score on the grid before any pick: {grid['exploit'].max():.3f}  "
      f"(request 2: 0.47);  largest exploration score: {grid['explore'].max():.3f}")
plan.drop(columns='why').round(3)

In [ ]:
fig = sp.plot_scores(df, plan, grid, path=FIG_DIR / 'm3_request3_scores.png', title='Oracle request 3')

Observation: _..._

## 3. Freeze every model's prediction (hold-out protocol), then write the request file

In [ ]:
models = {'quadratic': QuadraticSurrogate(), 'RBF thin-plate exact': RBFSurrogate('thin_plate_spline', 0.0),
          'RBF cubic': RBFSurrogate('cubic', 0.0), 'GP (ARD + nugget)': GPSurrogate()}
frozen = freeze_predictions(models, X, y, plan.D_m.values, plan.T_N.values, make_features,
                            RES_DIR / 'm3_request3_frozen_predictions.csv')
frozen.index = plan.index
frozen.insert(2, 'category', plan.category.values)
display(frozen.round(4))

req = write_request_csv(plan, ORACLE_DIR / 'request_3.csv', student_id=STUDENT_ID)
plan.round(4).to_csv(RES_DIR / 'm3_request3_plan.csv')
print(req.to_string(index=False))
print(f"\nEmail subject:  ED6002 ORACLE {STUDENT_ID} 3")

When the result comes back, save it as `data/oracle/result_3.csv`, then **first** compare it with
`results/m3_request3_frozen_predictions.csv` (hold-out RMSE per model), and only then refit on all 30 runs.

## 4. Defence (own words, for the decision log)
1. Why 2/3/1 and no repeat: _..._
2. Exploitation picks and what their small scores mean: _..._
3. Exploration picks (cliff region): _..._
4. Random pick: _..._